# Non-Normalizable Boltzmann Distributions

This notebook is a Python rebuild of an older class project. The original MATLAB code is gone, so the goal here is to reconstruct the main modeling ideas in a form that is easier to read, rerun, and revise.

There are two main pieces:

1. A Brownian particle moving in an asymptotically flat, i.e. 
$$
\lim_{x\to \infty} V(x)=0
$$ 
potential near a charged wall.

2. A near-wall diffusion model where the diffusivity depends on position.

The notebook is written in a teaching style: short markdown explanations, compact helper functions, and a small set of figures that can also be used on the portfolio site.


## Background and modeling idea

For a standard Boltzmann distribution we expect a density of the form

$$p(x) \propto e^{-V(x)/(k_B T)}.$$

When the partition function diverges, this expression is not a normalizable stationary density on its own. For asymptotically flat potentials, one instead expects a long-time scaling form where a Gaussian cutoff appears at finite time. In the project write-up this was written as

$$P_t(x) \approx \frac{1}{\sqrt{\pi D t}}\exp\!\left(-\frac{V(x)}{k_B T} - \frac{x^2}{4Dt}\right).$$

The specific potential used here is the exponential wall potential

$$V(x) = V_0 e^{-x/\ell_d},$$

which is a simple model for repulsion from a charged surface in a liquid.

After that, we look at a second model in which the diffusivity itself depends on position. This is one route toward diffusing diffusivity: the MSD can remain close to linear in time while the displacement distribution is still non-Gaussian.


In [3]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


In [4]:
def locate_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "index.html").exists() and (candidate / "projects").exists():
            return candidate
    return Path.cwd().resolve()


PROJECT_ROOT = locate_project_root()
IMAGE_DIR = PROJECT_ROOT / "projects" / "non-normalizable-boltzmann" / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

K_B = 1.380649e-23
TEMPERATURE = 298.15


@dataclass(frozen=True)
class PhysicalParams:
    temperature: float = TEMPERATURE
    viscosity: float = 1.0e-3
    particle_radius: float = 2.5e-6
    debye_length: float = 0.08e-6
    wall_potential_scale: float = 16.0

    @property
    def kbt(self) -> float:
        return K_B * self.temperature

    @property
    def gamma(self) -> float:
        return 6.0 * np.pi * self.viscosity * self.particle_radius

    @property
    def D0(self) -> float:
        return self.kbt / self.gamma

    @property
    def V0(self) -> float:
        return self.wall_potential_scale * self.kbt


params = PhysicalParams()
print(f"Project root: {PROJECT_ROOT}")
print(f"D0 = {params.D0 * 1e12:.4f} micrometers^2/s")
print(f"gamma = {params.gamma:.4e} N s / m")


Project root: /Users/anthonycaine/GitThis
D0 = 0.0874 micrometers^2/s
gamma = 4.7124e-08 N s / m


## Building the model in pieces

Instead of placing all helper functions in one large cell, we separate them by role in the model. Each code cell below corresponds to a specific mathematical ingredient: the potential, the boundary rule at the wall, the long-time scaling profile, the constant-diffusivity simulation, the near-wall diffusivity laws, and finally the variable-diffusivity simulation.


### Potential and force

The potential used in the write-up is

$$V(x) = V_0 e^{-x/\ell_d}.$$

The force is the negative derivative of the potential,

$$F(x) = -V'(x).$$

These functions appear in both the constant-diffusivity Langevin model and the position-dependent diffusivity model.


In [ ]:
def potential(x: np.ndarray, params: PhysicalParams) -> np.ndarray:
    return params.V0 * np.exp(-x / params.debye_length)


def potential_gradient(x: np.ndarray, params: PhysicalParams) -> np.ndarray:
    return -(params.V0 / params.debye_length) * np.exp(-x / params.debye_length)


def force(x: np.ndarray, params: PhysicalParams) -> np.ndarray:
    return -potential_gradient(x, params)


### Reflecting wall at the lower boundary

The particle represents distance from a wall, so values below the wall are not physically allowed. Numerically, if a time step tries to push a particle below the lower boundary, we reflect it back into the allowed region. In symbols, if the wall is at `lower` and a point lands below it, we mirror its overshoot across the wall.


In [ ]:
def reflect_lower(values: np.ndarray, lower: float) -> np.ndarray:
    reflected = values.copy()
    mask = reflected < lower
    reflected[mask] = lower + np.abs(reflected[mask] - lower)
    return reflected


### Long-time scaling profile

For asymptotically flat potentials, the write-up uses the finite-time scaling form

$$P_t(x) \approx \frac{1}{\sqrt{\pi D t}}\exp\!\left(-\frac{V(x)}{k_B T} - \frac{x^2}{4Dt}\right).$$

The function below evaluates that profile. Later we compare its shape to the histogram coming from the simulated particle ensemble.


In [ ]:
def scaling_solution(x: np.ndarray, t: float, params: PhysicalParams) -> np.ndarray:
    return (
        1.0 / np.sqrt(np.pi * params.D0 * t)
        * np.exp(-potential(x, params) / params.kbt - x**2 / (4.0 * params.D0 * t))
    )


### Constant-diffusivity Langevin stepper

The first simulation uses the overdamped Langevin model

$$dX_t = \frac{F(X_t)}{\gamma}dt + \sqrt{2D_0}\,dW_t.$$ 

In Euler-Maruyama form, each step is:

$$X_{n+1} \approx X_n + \frac{F(X_n)}{\gamma}\Delta t + \sqrt{2D_0\Delta t}\,\xi_n,$$

where \(\xi_n\) is a standard normal random variable. This is the code that creates the constant-diffusivity trajectories used in the first experiment.


In [ ]:
def simulate_constant_diffusivity(
    *, params: PhysicalParams, n_particles: int, n_steps: int, dt: float, seed: int
) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    positions = np.zeros(n_particles, dtype=float)
    history = np.empty((n_steps + 1, n_particles), dtype=float)
    history[0] = positions
    sigma = np.sqrt(2.0 * params.D0 * dt)

    for step in range(1, n_steps + 1):
        drift = force(positions, params) / params.gamma
        positions = positions + drift * dt + sigma * rng.standard_normal(n_particles)
        positions = reflect_lower(positions, 0.0)
        history[step] = positions

    times = np.arange(n_steps + 1) * dt
    return times, history


def simulate_sample_trajectories(
    *, params: PhysicalParams, n_particles: int, n_steps: int, dt: float, seed: int
):
    return simulate_constant_diffusivity(
        params=params,
        n_particles=n_particles,
        n_steps=n_steps,
        dt=dt,
        seed=seed,
    )


### Ensemble-average drift diagnostic

The write-up checks the average velocity against

$$\left\langle \frac{\Delta x}{\Delta t} \right\rangle = \frac{F(x)}{\gamma}.$$ 

The helper below bins simulated positions and averages the empirical velocity in each bin. That lets us compare the simulated ensemble drift to the theoretical drift curve.


In [ ]:
def binned_velocity(positions: np.ndarray, dt: float, bins: np.ndarray):
    x_prev = positions[:-1].ravel()
    x_next = positions[1:].ravel()
    velocity = (x_next - x_prev) / dt

    counts, edges = np.histogram(x_prev, bins=bins)
    weighted, _ = np.histogram(x_prev, bins=bins, weights=velocity)
    centers = 0.5 * (edges[:-1] + edges[1:])

    means = np.full_like(centers, np.nan, dtype=float)
    valid = counts > 75
    means[valid] = weighted[valid] / counts[valid]
    return centers, means, counts


### Near-wall diffusivity laws

For the second model we need explicit formulas for the position-dependent diffusivity \(D(x)\) and its derivative \(D'(x)\). The notebook uses two versions from the project notes:

- a Brenner-style approximation,
- a rational near-wall approximation.

These functions supply the local diffusivity needed in the variable-diffusivity stochastic differential equation.


In [ ]:
def diffusivity_brenner(z: np.ndarray, params: PhysicalParams):
    r = params.particle_radius
    D = params.D0 * (1.0 - 9.0 * r / (8.0 * z))
    dD = params.D0 * (9.0 * r) / (8.0 * z**2)
    return D, dD


def diffusivity_rational(z: np.ndarray, params: PhysicalParams):
    r = params.particle_radius
    numerator = 6.0 * z**2 + 2.0 * r * z
    denominator = 6.0 * z**2 + 9.0 * r * z + 2.0 * r**2
    numerator_prime = 12.0 * z + 2.0 * r
    denominator_prime = 12.0 * z + 9.0 * r

    D = params.D0 * numerator / denominator
    dD = params.D0 * (numerator_prime * denominator - numerator * denominator_prime) / denominator**2
    return D, dD


### Variable-diffusivity Langevin stepper

The second simulation is based on the It\^o overdamped model

$$dX_t = \left(D'(X_t) - \frac{D(X_t)}{k_B T}V'(X_t)\right)dt + \sqrt{2D(X_t)}\,dW_t.$$ 

So each numerical step combines three ideas:

- drift from the spatial variation of diffusivity,
- drift from the potential field,
- noise with size set by the local diffusivity.

The helper below runs many particles, then returns the time grid together with the second and fourth centered moments used later for MSD and kurtosis.


In [ ]:
def summarize_variable_diffusivity(
    *,
    params: PhysicalParams,
    n_particles: int,
    n_steps: int,
    dt: float,
    x0: float,
    diffusivity_model,
    lower_wall: float,
    seed: int,
):
    rng = np.random.default_rng(seed)
    positions = np.full(n_particles, x0, dtype=float)
    times = np.arange(n_steps + 1) * dt
    variance = np.zeros(n_steps + 1, dtype=float)
    fourth = np.zeros(n_steps + 1, dtype=float)

    for step in range(1, n_steps + 1):
        D, dD = diffusivity_model(positions, params)
        D = np.maximum(D, params.D0 * 1.0e-6)
        drift = dD - D * potential_gradient(positions, params) / params.kbt
        noise = np.sqrt(2.0 * D * dt) * rng.standard_normal(n_particles)
        positions = positions + drift * dt + noise
        positions = reflect_lower(positions, lower_wall)

        displacement = positions - x0
        centered = displacement - displacement.mean()
        variance[step] = np.mean(centered**2)
        fourth[step] = np.mean(centered**4)

    return times, variance, fourth


## Experiment 1: constant diffusivity with an exponential wall potential

We first test the simpler overdamped Langevin model

$$dX_t = \frac{F(X_t)}{\gamma}dt + \sqrt{2D_0}\,dW_t,$$

where the force comes from the potential \(V(x)=V_0 e^{-x/\ell_d}\).

The first plot compares the simulated position histogram at a large final time to the shape of the scaling solution. The second plot checks whether the ensemble drift agrees with \(F(x)/\gamma\), which is the deterministic drift predicted by the overdamped model.


In [ ]:
seed = 522
times, history = simulate_constant_diffusivity(
    params=params,
    n_particles=5000,
    n_steps=900,
    dt=1.0e-4,
    seed=seed,
)

final_time = times[-1]
final_positions = history[-1]
max_x = np.quantile(final_positions, 0.997)

bins = np.linspace(0.0, max_x, 70)
hist_density, edges = np.histogram(final_positions, bins=bins, density=True)
centers = 0.5 * (edges[:-1] + edges[1:])
profile = scaling_solution(centers, final_time, params)
amplitude = np.dot(hist_density, profile) / np.dot(profile, profile)

grid = np.linspace(0.0, max_x, 500)
theory = amplitude * scaling_solution(grid, final_time, params)

fig, ax = plt.subplots(figsize=(8.5, 5.4))
ax.hist(
    final_positions * 1.0e6,
    bins=edges * 1.0e6,
    density=True,
    alpha=0.65,
    color="#b55233",
    label="Simulated ensemble at final time",
)
ax.plot(grid * 1.0e6, theory / 1.0e6, color="#1d2733", lw=2.3, label="Best-fit scaling profile")
ax.set_xlabel("Distance from wall (micrometers)")
ax.set_ylabel("Probability density (1 / micrometer)")
ax.set_title("Non-normalizable Boltzmann scaling solution")
ax.grid(alpha=0.18)
ax.legend()
plt.show()

fig.savefig(IMAGE_DIR / "nnbd_scaling_solution.png", dpi=180, bbox_inches="tight")

bins = np.linspace(0.0, 0.7e-6, 60)
centers, measured, counts = binned_velocity(history, dt=times[1] - times[0], bins=bins)
theory_velocity = force(centers, params) / params.gamma
valid = counts > 250

fig, ax = plt.subplots(figsize=(8.5, 5.4))
ax.plot(centers[valid] * 1.0e6, theory_velocity[valid] * 1.0e6, color="#1d2733", lw=2.3, label="F(x) / gamma")
ax.plot(centers[valid] * 1.0e6, measured[valid] * 1.0e6, color="#b55233", lw=1.9, label="Mean simulated velocity")
ax.set_xlabel("Distance from wall (micrometers)")
ax.set_ylabel("Velocity (micrometers / second)")
ax.set_title("Ensemble drift matches the overdamped Langevin drift")
ax.set_ylim(bottom=0.0)
ax.grid(alpha=0.18)
ax.legend()
plt.show()

fig.savefig(IMAGE_DIR / "nnbd_velocity_validation.png", dpi=180, bbox_inches="tight")


### A few sample particle paths

The ensemble plots are useful, but they can feel abstract. The next figure tracks a small handful of individual particles. This makes the near-wall behavior easier to read: the wall prevents penetration below zero, the repulsive force pushes particles away when they get too close, and thermal noise keeps the paths irregular.


In [ ]:
traj_times, traj_history = simulate_sample_trajectories(
    params=params,
    n_particles=6,
    n_steps=1800,
    dt=5.0e-5,
    seed=seed + 100,
)

fig, ax = plt.subplots(figsize=(8.5, 5.4))
colors = ["#1d2733", "#b55233", "#2b6f89", "#8d6a9f", "#5b8a5a", "#cc7a00"]

for idx in range(traj_history.shape[1]):
    ax.plot(
        traj_times,
        traj_history[:, idx] * 1.0e6,
        lw=1.8,
        alpha=0.95,
        color=colors[idx % len(colors)],
        label=f"Particle {idx + 1}",
    )

ax.axhline(0.0, color="k", ls="--", lw=1.2, alpha=0.7, label="Wall")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Distance from wall (micrometers)")
ax.set_title("Sample particle trajectories near the charged wall")
ax.grid(alpha=0.18)
ax.legend(ncol=2, fontsize=9)
plt.show()

fig.savefig(IMAGE_DIR / "nnbd_particle_trajectories.png", dpi=180, bbox_inches="tight")


## Experiment 2: position-dependent diffusivity near the wall

Now we move to the near-wall model discussed in the project write-up. The It\^o overdamped model is

$$dX_t = \left(D'(X_t) - \frac{D(X_t)}{k_B T}V'(X_t)\right)dt + \sqrt{2D(X_t)}\,dW_t.$$ 

Two diffusivity laws are tested here:

- a Brenner-style near-wall approximation,
- a rational approximation used in the original project notes.

The first figure checks whether the displacement variance remains roughly linear in time. The second tracks the excess kurtosis, which is zero for a Gaussian displacement law.


In [ ]:
start_positions = [2.0 * params.particle_radius, 10.0 * params.particle_radius]
lower_wall = 1.15 * params.particle_radius
model_specs = [
    ("Brenner approximation", diffusivity_brenner, "#b55233"),
    ("Rational near-wall model", diffusivity_rational, "#2b6f89"),
]

msd_fig, msd_axes = plt.subplots(1, 2, figsize=(12.5, 5.2), sharey=True)
kurt_fig, kurt_ax = plt.subplots(figsize=(8.5, 5.4))

for axis, (title, model, color) in zip(msd_axes, model_specs):
    for idx, x0 in enumerate(start_positions):
        times, var, fourth = summarize_variable_diffusivity(
            params=params,
            n_particles=3500,
            n_steps=1500,
            dt=2.0e-3,
            x0=x0,
            diffusivity_model=model,
            lower_wall=lower_wall,
            seed=seed + idx + (0 if model is diffusivity_brenner else 10),
        )

        label = f"x0 = {x0 / params.particle_radius:.0f} r"
        axis.plot(times, var * 1.0e12, lw=2.0, label=label)

        if idx == 0:
            kurtosis = np.divide(
                fourth,
                var**2,
                out=np.full_like(var, np.nan),
                where=var > 0.0,
            )
            kurt_ax.plot(times, kurtosis - 3.0, lw=2.0, color=color, label=title)

    axis.plot(times, 2.0 * params.D0 * times * 1.0e12, "k--", lw=1.6, label="2 D0 t")
    axis.set_title(title)
    axis.set_xlabel("Time (seconds)")
    axis.grid(alpha=0.18)
    axis.legend()

msd_axes[0].set_ylabel("Variance of displacement (micrometers squared)")
msd_fig.suptitle("Position-dependent diffusivity still gives near-linear MSD growth", y=1.02)
plt.tight_layout()
plt.show()

msd_fig.savefig(IMAGE_DIR / "diffusing_diffusivity_msd.png", dpi=180, bbox_inches="tight")

kurt_ax.axhline(0.0, color="k", ls="--", lw=1.4, label="Gaussian baseline")
kurt_ax.set_xlabel("Time (seconds)")
kurt_ax.set_ylabel("Excess kurtosis")
kurt_ax.set_title("Displacement kurtosis remains non-Gaussian near the wall")
kurt_ax.grid(alpha=0.18)
kurt_ax.legend()
plt.show()

kurt_fig.savefig(IMAGE_DIR / "diffusing_diffusivity_kurtosis.png", dpi=180, bbox_inches="tight")


## Conclusion

This rebuild gives a workable Python version of the original project ideas.

- In the constant-diffusivity case, the simulated ensemble has the expected broad shape associated with the finite-time scaling profile.
- The measured ensemble drift agrees well with the overdamped Langevin drift term.
- In the varying-diffusivity models, the MSD remains close to linear while the excess kurtosis can stay away from zero for long time windows.

That does not prove the model is the only correct discretization, but it does give a clean starting point for further experimentation.


## References and project notes

This notebook follows the themes and notation of the original course write-up. The write-up referenced several strands of literature:

- Non-normalizable Boltzmann states and infinite-ergodic ideas: `[NNB1]`, `[NNB2]`, `[RBG]`
- Diffusing diffusivity and near-wall experiments: `[Chubynsky]`, `[NWC]`, `[lau]`
- Energy landscapes and heterogeneous diffusion in biological settings: `[EDincell]`
- Near-wall mobility correction formulas: `[BRENNER1961242]`

Since this notebook is being reconstructed from the old project rather than from the original BibTeX database, I have kept the reference labels from the write-up instead of inventing new citation metadata.
